# SD3.5 mask-FREE EDIT LoRA — all-in-one (train → test → contact sheet)

IP2P/PIPE-style object addition: the model adds a person from the **source image
+ instruction only** — NO mask, NO hard-restore. It decides where/scale/pose from
context (per the PIPE paper, Wasserman et al. 2404.18212).

Pipeline: setup → SD3.5 access + build PIPE eval → train (LoRA + input_proj,
zip immediately) → eval in a fresh subprocess → contact sheet (source | edit | target).

Trains TWO artifacts (LoRA + input_proj.pt), both required at inference. To run
unattended set `SMOKE = False` then **Save Version → Save & Run All**. Needs GPU +
SD3.5 access (HF_TOKEN secret or a mounted model dataset).

> Mask-free tradeoff: there is no mask to fit and background is NOT guaranteed
> byte-exact — the model owns the whole image.

## 1. Setup — install pinned stack (batch-safe, no kernel restart)

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2','datasets>=2.20',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, diffusers, torch
print('transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert transformers.__version__ == '4.46.3', f'transformers is {transformers.__version__}, expected 4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU'
print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access (gated) + build the PIPE eval set

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local); print('local SD3.5 mount:', SD35_MODEL)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'SD3.5 is gated. Add a Kaggle secret HF_TOKEN or mount the model dataset.'
    from huggingface_hub import login; login(token=HF_TOKEN); print('HF login OK')

In [ ]:
from LoRA.data.build_eval_cases_pipe import run_build_pipe_eval
import json
WORK = Path('/kaggle/working/vin_lora')
EVAL = WORK/'eval'/'pipe_eval_v1'
EVAL_LIMIT = 12   # for a fast contact sheet; raise for a fuller eval
if not (EVAL/'cases.jsonl').exists():
    run_build_pipe_eval(WORK, eval_set='pipe_eval_v1', split='test', person_only=True, limit=EVAL_LIMIT)
cases = [json.loads(l) for l in (EVAL/'cases.jsonl').read_text().splitlines() if l.strip()]
print(len(cases), 'eval cases')

## 3. Train the mask-free edit-LoRA on PIPE pairs

Smoke first (200 steps / 200 samples). For a real adapter set `SMOKE = False`
(config: 4000 steps / 4000 samples, grad-accum 16, lr 5e-5 — much longer).

In [ ]:
from LoRA.train.train_maskfree_edit import run_training
import shutil
SMOKE = True   # True = 200 steps/200 samples (fast); False = full (4000/4000)
kw = dict(max_train_steps=200, num_train_samples=200) if SMOKE else {}
train = run_training(WORK, base_model_id=SD35_MODEL, hf_token=HF_TOKEN, **kw)
RUN_DIR = train['run_dir']
print('trained ->', RUN_DIR)
ZIP = shutil.make_archive(str(RUN_DIR), 'zip', str(RUN_DIR))
print('adapter zip ->', ZIP, '(download this / make it a dataset)')
train['provenance']

## 4. Eval in a FRESH process (avoids train+reload RAM OOM)

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'LoRA.inference.run_maskfree_eval',
       '--run-dir', str(RUN_DIR), '--eval-dir', str(EVAL),
       '--base-model', str(SD35_MODEL), '--steps', '30']
if HF_TOKEN:
    cmd += ['--hf-token', HF_TOKEN]
print('running eval subprocess...')
r = subprocess.run(cmd, cwd='/kaggle/working/VIN')
print('eval exit code:', r.returncode)
EVAL_OUT = RUN_DIR / 'maskfree_eval'

## 5. Show the contact sheet (source | edit | target)

In [ ]:
from pathlib import Path
sheet = EVAL_OUT / 'contact_sheet.png'
assert sheet.exists(), f'eval did not produce a contact sheet (check subprocess output above): {sheet}'
print('contact sheet:', sheet)
from IPython.display import Image as IPImage, display
display(IPImage(str(sheet)))